# Benchmark de documentação — Kaggle

Executa 100 exemplos por linguagem com Qwen3 1.7B ou Ministral 3 3B.

Antes de executar:

1. Use um **Kaggle Notebook comum**, não Benchmark Task.
2. Em **Settings**, selecione uma GPU T4 ou P100.
3. Ative **Internet**.
4. Em **Add Input**, envie `Celx-colab-v8.zip` como dataset privado.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

assert Path('/kaggle/input').exists(), 'Este notebook deve ser executado no Kaggle.'
try:
    import torch
except ModuleNotFoundError as error:
    raise RuntimeError(
        'PyTorch ausente. Crie um Kaggle Notebook comum e ative a GPU em Settings.'
    ) from error
assert torch.cuda.is_available(), 'Ative uma GPU em Settings → Accelerator.'
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GB')

## Localizar e extrair o projeto

O código procura qualquer ZIP adicionado em `/kaggle/input` que contenha `configs/model_candidates.yaml`.

In [ ]:
import os
import zipfile

input_root = Path('/kaggle/input')
working_root = Path('/kaggle/working')
candidate_archives = []
for archive in input_root.rglob('*.zip'):
    try:
        with zipfile.ZipFile(archive) as zipped:
            if 'configs/model_candidates.yaml' in zipped.namelist():
                candidate_archives.append(archive)
    except zipfile.BadZipFile:
        pass
assert candidate_archives, (
    'Celx-colab-v8.zip não encontrado. Adicione-o pelo botão Add Input.'
)
archive = sorted(candidate_archives)[-1]
repo_dir = working_root / 'legacy-doc-project'
if repo_dir.exists():
    shutil.rmtree(repo_dir)
repo_dir.mkdir(parents=True)
shutil.unpack_archive(archive, repo_dir)
os.environ['PYTHONPATH'] = str(repo_dir)
print('Pacote:', archive)
print('Projeto:', repo_dir)

In [ ]:
%cd /kaggle/working/legacy-doc-project
%pip install -q "transformers>=4.51" "accelerate>=1.0" "bitsandbytes>=0.45" "datasets>=3.0" "PyYAML>=6.0"
%pip install -q -e .
print('Dependências instaladas. Se o Kaggle solicitar, reinicie a sessão e execute novamente.')

## Preparar os 400 casos

Se `expanded_100.jsonl` tiver sido adicionado como Input, ele será reutilizado. Caso contrário, o notebook baixa CodeXGLUE e Spider e constrói o benchmark.

In [ ]:
benchmark_path = repo_dir / 'dataset/benchmark/expanded_100.jsonl'
existing_benchmarks = [
    path for path in input_root.rglob('expanded_100.jsonl') if path.is_file()
]
if existing_benchmarks:
    benchmark_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(existing_benchmarks[0], benchmark_path)
    print('Benchmark reutilizado:', existing_benchmarks[0])
else:
    subprocess.run(
        [sys.executable, 'scripts/prepare_dataset.py'],
        check=True, env=os.environ.copy(),
    )
    subprocess.run(
        [sys.executable, 'scripts/analyze_dataset.py'],
        check=True, env=os.environ.copy(),
    )
    subprocess.run(
        [sys.executable, 'scripts/build_benchmark.py'],
        check=True, env=os.environ.copy(),
    )

import json
records = [json.loads(line) for line in benchmark_path.read_text(encoding='utf-8').splitlines()]
counts = {language: sum(row['language'] == language for row in records) for language in ('python', 'php', 'javascript', 'sql')}
assert counts == {'python': 100, 'php': 100, 'javascript': 100, 'sql': 100}
print('Benchmark:', counts)

## Executar um modelo

Execute somente um modelo por sessão. O Qwen está selecionado por padrão. A célula reaproveita resultados existentes em `/kaggle/working`.

In [ ]:
executar_qwen = True  # @param {type:"boolean"}
executar_ministral = False  # @param {type:"boolean"}
selecionados = [
    nome for nome, ativo in {
        'qwen3-1.7b': executar_qwen,
        'ministral3-3b': executar_ministral,
    }.items() if ativo
]
assert len(selecionados) == 1, 'Selecione exatamente um modelo.'
subprocess.run(
    [sys.executable, 'scripts/run_baseline.py', '--model', selecionados[0]],
    check=True, env=os.environ.copy(),
)

In [ ]:
results_dir = repo_dir / 'outputs/benchmark_100'
for result in sorted(results_dir.glob('*.jsonl')):
    completed = sum(1 for line in result.open(encoding='utf-8') if line.strip())
    print(f'{result.stem}: {completed}/400 respostas')
subprocess.run(
    [sys.executable, 'scripts/compare_models.py', '--results', str(results_dir), '--output', str(results_dir / 'model_comparison.csv')],
    check=True, env=os.environ.copy(),
)
import pandas as pd
display(pd.read_csv(results_dir / 'model_comparison.csv'))

## Gerar arquivo para download

Após a execução, escolha **Save Version** para preservar `/kaggle/working`. O ZIP também aparecerá no painel Output.

In [ ]:
output_archive = shutil.make_archive(
    '/kaggle/working/celx-benchmark-results',
    'zip',
    root_dir=results_dir,
)
print('Resultados:', output_archive)
print('Use o painel Output para baixar o ZIP ou salve uma versão do notebook.')